# Product Embeddings for Content-Based Recommendation

Goal: Generate semantic representations of products using product metadata.

Why: Collaborative filtering relies on historical interactions and struggles with cold-start products.

Content-based embeddings allow recommendations based on product meaning rather than user behavior.

Output: One dense embedding vector for each product.

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from tqdm.auto import tqdm

In [2]:
# Load Product Catalog
catalog = pd.read_parquet(
    "../data/catalog.parquet"
)

catalog.shape

(63001, 6)

# Inspect Product Text

Goal

Verify that product descriptions contain useful semantic information before generating embeddings.

In [ ]:
catalog[
    [
        "asin",
        "title",
        "product_text"
    ]
].sample(
    3,
    random_state=42
)

,asin,title,product_text
51834,B008OPT8WO,"12 cell, 8800mAh Extended Hight Capacity Lapto...","12 cell, 8800mAh Extended Hight Capacity Lapto..."
4148,B000234498,C2G / Cables to Go - 27028 - Firewire 6-Pin Fe...,C2G / Cables to Go - 27028 - Firewire 6-Pin Fe...
48192,B007M5NKRQ,"6 Pair - 12 Pcs - Size Medium, of Memory Foam ...","6 Pair - 12 Pcs - Size Medium, of Memory Foam ..."


# Load Sentence Transformer

Goal: Use a pretrained transformer model to generate dense semantic representations of products.

Model: all-MiniLM-L6-v2

Output Dimension: 384

In [4]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Generate Product Embeddings

Goal

Convert each product into a 384-dimensional semantic vector.

These vectors will later support:

- Similar product retrieval
- Cold-start recommendation
- Hybrid recommendation

In [5]:
texts = catalog[
    "product_text"
].tolist()

texts = [
    str(t)[:2000]
    for t in texts
]

embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/493 [00:00<?, ?it/s]

# Validate Embeddings

In [6]:
embeddings.shape

(63001, 384)

In [7]:
np.save(
    "product_embeddings.npy",
    embeddings
)

In [8]:
catalog_embeddings = catalog[
    [
        "asin",
        "title"
    ]
].copy()

catalog_embeddings.to_parquet(
    "catalog_embeddings.parquet",
    index=False
)

# Semantic Product Similarity

Goal: Retrieve products that are semantically similar according to transformer embeddings.

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def similar_products(
    asin,
    top_k=10
):
    
    idx = catalog[
        catalog["asin"] == asin
    ].index[0]
    
    sims = cosine_similarity(
        embeddings[idx].reshape(1,-1),
        embeddings
    )[0]
    
    top_idx = np.argsort(
        sims
    )[::-1][1:top_k+1]
    
    return catalog.iloc[
        top_idx
    ][
        ["asin","title"]
    ]

In [10]:
sample_asin = catalog.iloc[450]["asin"]

similar_products(
    sample_asin,
    top_k=10
)

,asin,title
452,B00004WHFL,Franklin EBM-901 eBookman (Metallic Black)
35136,B004DHB7KI,PocketBook 701R 701 Red
31387,B003UE4A86,Kobo Wifi eReader
32402,B003Z99AC2,Sony Reader Touch Edition - Red (PRS650RC)
16938,B001AAOZHI,Livescribe 1 GB Pulse Smartpen (APA-00001)
13926,B000WP2RC2,Sony PRS-505/LC Blue Digital Book Reader
58665,B00CN6LTBI,Anybest Digital Mini Portable rechargeable Aud...
24371,B002MSHQ46,Sony Digital Reader Touch Edition - Red (PRS60...
13933,B000WPXQ2M,Sony PRS-505 Portable Digital e-Reader System ...
1041,B00005T3UH,RCA eBook Reader (REB1100)


# TEST

In [11]:
def show_similar_products(
    asin,
    top_k=10
):
    
    idx = catalog[
        catalog["asin"] == asin
    ].index[0]

    sims = cosine_similarity(
        embeddings[idx].reshape(1,-1),
        embeddings
    )[0]

    top_idx = np.argsort(
        sims
    )[::-1][1:top_k+1]

    query_title = catalog.iloc[idx]["title"]

    print("\nQUERY PRODUCT")
    print("-" * 60)
    print(query_title)

    print("\nSIMILAR PRODUCTS")
    print("-" * 60)

    for rank, i in enumerate(top_idx, 1):

        print(
            f"{rank:2d}. "
            f"{catalog.iloc[i]['title']}"
        )

In [ ]:
sample_asin1 = catalog.iloc[700]["asin"]

show_similar_products(
    sample_asin1
)


QUERY PRODUCT
------------------------------------------------------------
Netgear RT314 Internet Access Router

SIMILAR PRODUCTS
------------------------------------------------------------
 1. Netgear MR314 802.11b Wireless Cable/DSL Router with 4-Port Switch
 2. NETGEAR MA311 - Network adapter - PCI - 802.11b
 3. Linkskey 4-Port DSL/Cable IP Sharing Router Wired (LKR-604)
 4. Linksys WRT54GL Wireless-G Broadband Router
 5. Cisco-Linksys WRT54G2 Wireless-G Broadband Router
 6. Cisco-Linksys Compact Wireless-G Broadband Router WRT54GC
 7. Netgear R6200 AC1200 802.11ac Dual Band Gigabit WiFi Router
 8. Linksys Network Everywhere 4-Port Cable/DSL Router
 9. NETGEAR RangeMax Dual Band Wireless-N Gigabit Router WNDR3700 - Wireless router - 4-port switch - Gigabit Ethernet - 802.11 a/b/g/n (draft 2.0) - desktop RANGEMAX DUAL BAND WLS N GETH RTR Manufacturer Part Number WNDR3700-100NAS
10. Cisco-Linksys Wireless-G Cable Gateway (WCG200)
